In [ ]:
import requests
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
CLIENT_ID = os.getenv("BLIZZARD_CLIENT_ID")
CLIENT_SECRET = os.getenv("BLIZZARD_CLIENT_SECRET")

token_response = requests.post(
    "https://oauth.battle.net/token",
    data={"grant_type": "client_credentials"},
    auth=(CLIENT_ID, CLIENT_SECRET)
)
access_token = token_response.json()["access_token"]
print("Token received.")

Token received!


In [ ]:

# Fetch all Hearthstone cards from the Blizzard API
all_cards = []
page = 1
url = "https://us.api.blizzard.com/hearthstone/cards"
headers = {"Authorization": f"Bearer {access_token}"}

while True:
    response = requests.get(url, headers=headers, params={"locale": "en_US", "pageSize": 100, "page": page})
    data = response.json()
    cards = data.get("cards", [])
    if not cards:
        break
    all_cards.extend(cards)
    print(f"Page {page} fetched — {len(all_cards)} cards total")
    if page >= data["pageCount"]:
        break
    page += 1

Page 1 fetched — 100 cards total
Page 2 fetched — 200 cards total
Page 3 fetched — 300 cards total
Page 4 fetched — 400 cards total
Page 5 fetched — 500 cards total
Page 6 fetched — 600 cards total
Page 7 fetched — 700 cards total
Page 8 fetched — 800 cards total
Page 9 fetched — 900 cards total
Page 10 fetched — 1000 cards total
Page 11 fetched — 1100 cards total
Page 12 fetched — 1200 cards total
Page 13 fetched — 1300 cards total
Page 14 fetched — 1400 cards total
Page 15 fetched — 1500 cards total
Page 16 fetched — 1600 cards total
Page 17 fetched — 1700 cards total
Page 18 fetched — 1800 cards total
Page 19 fetched — 1900 cards total
Page 20 fetched — 2000 cards total
Page 21 fetched — 2100 cards total
Page 22 fetched — 2200 cards total
Page 23 fetched — 2300 cards total
Page 24 fetched — 2400 cards total
Page 25 fetched — 2500 cards total
Page 26 fetched — 2600 cards total
Page 27 fetched — 2700 cards total
Page 28 fetched — 2800 cards total
Page 29 fetched — 2900 cards total
Pag

In [32]:
hearthstone_df = pd.DataFrame(all_cards)
# Fetch metadata
metadata_response = requests.get(
    "https://us.api.blizzard.com/hearthstone/metadata",
    headers=headers,
    params={"locale": "en_US"}
)
metadata = metadata_response.json()

#Using metadata api to map ids to names for easier reading
class_map = {c["id"]: c["slug"] for c in metadata["classes"]}

hearthstone_df["className"] = hearthstone_df["classId"].map(class_map)
set_map = {s["id"]: s["slug"] for s in metadata["sets"]}
spell_school_map = {s["id"]: s["slug"] for s in metadata["spellSchools"]}
class_map = {c["id"]: c["slug"] for c in metadata["classes"]}

hearthstone_df["className"] = hearthstone_df["classId"].map(class_map)
hearthstone_df["setName"] = hearthstone_df["cardSetId"].map(set_map)
hearthstone_df["spellSchoolName"] = hearthstone_df["spellSchoolId"].map(spell_school_map)

minion_type_map = {m["id"]: m["slug"] for m in metadata["minionTypes"]}
rarity_map = {r["id"]: r["slug"] for r in metadata["rarities"]}
card_type_map = {t["id"]: t["slug"] for t in metadata["types"]}
hearthstone_df["minionTypeName"] = hearthstone_df["minionTypeId"].map(minion_type_map)
hearthstone_df["rarityName"] = hearthstone_df["rarityId"].map(rarity_map)
hearthstone_df["cardTypeName"] = hearthstone_df["cardTypeId"].map(card_type_map)


hearthstone_df["multiClassNames"] = hearthstone_df["multiClassIds"].apply(
    lambda ids: [class_map.get(i) for i in ids] if isinstance(ids, list) else [] )


